# GOEA Lost gene analysis

In [4]:
from Bio import SeqIO
import os
import pandas as pd

# Load Sequence index

In [5]:
proteome_path = '../data/proteomes/'

In [12]:
seqs = {}
fns = []
for fn in os.listdir(proteome_path):
    #recs = SeqIO.index(os.path.join(proteome_path, fn), 'fasta')
    fns.append(os.path.join(proteome_path, fn))

In [15]:
seqs = SeqIO.index_db(':memory:', filenames=fns, format='fasta')

Idea: for a particlar GO term, extract the HOGs for a particular dataset and get the closest outgroup gene. We then want to do a search against UniProt or NR.

In [47]:
go_df = pd.read_csv( '../fastoma_round2_tree_c_LG/eagle_results_with_ancestral_genes.tsv.gz', sep="\t")

In [45]:
go_df = go_df[go_df.p_fdr_bh < 0.05/62].set_index('GO_ID')

In [46]:
go_df.loc['GO:0016117']

,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,study_proportion,...,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term,branch_type,event_type,parent_name,child_name,branch_grouping
GO_ID,,,,,,,,,,,,,,,,,,,,,
GO:0016117,carotenoid biosynthetic process,BP,5,1.867795e-20,3.119218e-17,1.417826e-18,56,1535,56 / 1535,0.036482,...,13904,132 / 13904,0.009494,3.842780,HOG:0000509_21@C_LG.16;HOG:0000575_20@C_LG.16;...,internal,lost,C_LG.16,C_LG.29,SLL
GO:0016117,carotenoid biosynthetic process,BP,5,7.671479e-19,1.091651e-15,4.962052e-17,55,1637,55 / 1637,0.033598,...,13840,129 / 13840,0.009321,3.604627,HOG:0000509_20@C_LG.33;HOG:0000575_20@C_LG.33;...,internal,lost,C_LG.33,C_LG.34,SLC
GO:0016117,carotenoid biosynthetic process,BP,5,5.598512e-14,8.425760e-11,3.009200e-12,51,1726,51 / 1726,0.029548,...,13918,136 / 13918,0.009772,3.023899,HOG:0000509_20@SLC;HOG:0000575_20@SLC;HOG:0000...,internal,lost,SLC,C_LG.37,SLC
GO:0016117,carotenoid biosynthetic process,BP,5,5.312496e-10,8.181245e-07,1.152288e-08,39,1777,39 / 1777,0.021947,...,12512,99 / 12512,0.007912,2.773759,HOG:0000597_25@C_LG.20;HOG:0001079_25@C_LG.20;...,internal,lost,C_LG.20,C_LG.21,SLL
GO:0016117,carotenoid biosynthetic process,BP,5,6.811605e-07,3.916673e-04,8.514506e-06,56,2678,56 / 2678,0.020911,...,13634,153 / 13634,0.011222,1.863414,HOG:0000509_19@SLL_SLC_SP;HOG:0000575_19@SLL_S...,internal,gained,C_LG.15,SLL_SLC_SP,SLL_SLC_SP
GO:0016117,carotenoid biosynthetic process,BP,5,4.072394e-07,6.910853e-04,2.406405e-05,32,1174,32 / 1174,0.027257,...,13804,145 / 13804,0.010504,2.594889,HOG:0000575_19@SP;HOG:0000694_19@SP;HOG:000537...,internal,lost,SP,C_LG.45,SP
GO:0016117,carotenoid biosynthetic process,BP,5,2.133191e-05,3.515498e-02,4.882637e-04,26,1190,26 / 1190,0.021849,...,13599,124 / 13599,0.009118,2.396137,HOG:0000509_24@C_LG.19;HOG:0000575_24@C_LG.19;...,internal,lost,C_LG.19,C_LG.20,SLL


In [53]:
dfs = []
for (go_id, zdf) in go_df.groupby(['GO_ID', 'event_type']):
    zdf['p_fdr_bh_group_adjusted'] = zdf['p_fdr_bh'] * len(zdf)
    dfs.append(zdf)

In [55]:
zdf = pd.concat(dfs)

In [57]:
(zdf['p_fdr_bh_group_adjusted'] < 0.05).sum()

17222

In [59]:
(zdf['p_fdr_bh'] < 0.05).sum()

24861

In [61]:
zdf[zdf.GO_ID == 'GO:0016117']

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,...,population_ratio,population_proportion,fold_change,study_entries_with_go_term,branch_type,event_type,parent_name,child_name,branch_grouping,p_fdr_bh_group_adjusted
11938,GO:0016117,carotenoid biosynthetic process,BP,5,2.783950e-04,1.000000e+00,5.418467e-03,25,1227,25 / 1227,...,97 / 9795,0.009903,2.057445,HOG:0001202_5@C_LG.05;HOG:0002755_5@C_LG.05;HO...,internal,duplicated,C_LG.05,C_LG.06,outgroup,1.083693e-02
16982,GO:0016117,carotenoid biosynthetic process,BP,5,1.798951e-03,1.000000e+00,1.806802e-02,21,1038,21 / 1038,...,103 / 10004,0.010296,1.964981,HOG:0008619_11@C_LG.08;HOG:0008876_11@C_LG.08;...,internal,duplicated,C_LG.08,C_LG.54,outgroup,3.613604e-02
3930,GO:0016117,carotenoid biosynthetic process,BP,5,6.811605e-07,3.916673e-04,8.514506e-06,56,2678,56 / 2678,...,153 / 13634,0.011222,1.863414,HOG:0000509_19@SLL_SLC_SP;HOG:0000575_19@SLL_S...,internal,gained,C_LG.15,SLL_SLC_SP,SLL_SLC_SP,8.514506e-06
777,GO:0016117,carotenoid biosynthetic process,BP,5,1.867795e-20,3.119218e-17,1.417826e-18,56,1535,56 / 1535,...,132 / 13904,0.009494,3.842780,HOG:0000509_21@C_LG.16;HOG:0000575_20@C_LG.16;...,internal,lost,C_LG.16,C_LG.29,SLL,9.924784e-18
861,GO:0016117,carotenoid biosynthetic process,BP,5,7.671479e-19,1.091651e-15,4.962052e-17,55,1637,55 / 1637,...,129 / 13840,0.009321,3.604627,HOG:0000509_20@C_LG.33;HOG:0000575_20@C_LG.33;...,internal,lost,C_LG.33,C_LG.34,SLC,3.473437e-16
1328,GO:0016117,carotenoid biosynthetic process,BP,5,5.598512e-14,8.425760e-11,3.009200e-12,51,1726,51 / 1726,...,136 / 13918,0.009772,3.023899,HOG:0000509_20@SLC;HOG:0000575_20@SLC;HOG:0000...,internal,lost,SLC,C_LG.37,SLC,2.106440e-11
2145,GO:0016117,carotenoid biosynthetic process,BP,5,5.312496e-10,8.181245e-07,1.152288e-08,39,1777,39 / 1777,...,99 / 12512,0.007912,2.773759,HOG:0000597_25@C_LG.20;HOG:0001079_25@C_LG.20;...,internal,lost,C_LG.20,C_LG.21,SLL,8.066016e-08
4469,GO:0016117,carotenoid biosynthetic process,BP,5,4.072394e-07,6.910853e-04,2.406405e-05,32,1174,32 / 1174,...,145 / 13804,0.010504,2.594889,HOG:0000575_19@SP;HOG:0000694_19@SP;HOG:000537...,internal,lost,SP,C_LG.45,SP,1.684483e-04
7020,GO:0016117,carotenoid biosynthetic process,BP,5,2.133191e-05,3.515498e-02,4.882637e-04,26,1190,26 / 1190,...,124 / 13599,0.009118,2.396137,HOG:0000509_24@C_LG.19;HOG:0000575_24@C_LG.19;...,internal,lost,C_LG.19,C_LG.20,SLL,3.417846e-03
10701,GO:0016117,carotenoid biosynthetic process,BP,5,6.000421e-05,1.480904e-01,3.525962e-03,22,982,22 / 982,...,125 / 13874,0.009010,2.486582,HOG:0000575_20@C_LG.17;HOG:0000597_20@C_LG.17;...,internal,lost,C_LG.17,C_LG.28,SLL,2.468173e-02
